# 01 — Préparation du corpus

Projet GPT reproductible · Notebook en français.


## 1. Research objective

Question : comment nettoyer et partitionner les documents sans fuite ? Hypothèse : normaliser et dédupliquer avant un split déterministe rend le corpus plus fiable.


## 2. Configuration

Seed 2026; ratios train/validation/test = 0.8/0.1/0.1. Split au niveau document.


## 3. Implementation

CorpusProcessor normalise les espaces, filtre les courts et retire les doublons exacts SHA-256. DatasetSplitter vérifie les ratios.


In [1]:
import hashlib, random, re
from dataclasses import dataclass
CONFIG={"seed":2026,"ratios":(.8,.1,.1),"min_chars":1}
@dataclass(frozen=True)
class Report: total:int; kept:int; duplicates:int; short:int; mean_chars:float
class CorpusProcessor:
    @staticmethod
    def normalize(text):
        if not isinstance(text,str): raise TypeError("Document non textuel")
        return re.sub(r"\s+"," ",text.replace("\r\n","\n").replace("\r","\n")).strip()
    def process(self,docs,min_chars=1):
        out=[]; seen=set(); dup=short=0; total=0
        for raw in docs:
            total+=1; t=self.normalize(raw)
            if len(t)<min_chars: short+=1; continue
            h=hashlib.sha256(t.encode()).digest()
            if h in seen: dup+=1; continue
            seen.add(h); out.append(t)
        return out,Report(total,len(out),dup,short,sum(map(len,out))/len(out) if out else 0.)
class DatasetSplitter:
    def __init__(self,ratios=CONFIG['ratios'],seed=CONFIG['seed']):
        if any(x<0 for x in ratios) or abs(sum(ratios)-1)>1e-9: raise ValueError('Ratios invalides')
        self.ratios,self.seed=ratios,seed
    def split(self,docs):
        a=list(docs); random.Random(self.seed).shuffle(a); n=len(a); i=int(n*self.ratios[0]); j=int(n*self.ratios[1])
        return {'train':a[:i],'validation':a[i:i+j],'test':a[i+j:]}
docs=[' Un modèle\nprédit le token. ','Le tokenizer segmente.','L’attention combine le contexte.','Un modèle prédit le token.','  ']
clean,report=CorpusProcessor().process(docs,CONFIG['min_chars']); splits=DatasetSplitter().split(clean)
print(report,{k:len(v) for k,v in splits.items()})


Report(total=5, kept=3, duplicates=1, short=1, mean_chars=26.666666666666668) {'train': 2, 'validation': 0, 'test': 1}


## 4. Unit tests

Assertions sur nettoyage, déduplication, partitions disjointes et reproductibles.


In [2]:
p,r=CorpusProcessor().process([' a\n b ','a b','','z'])
assert p==['a b','z'] and (r.duplicates,r.short)==(1,1)
a=DatasetSplitter((.5,.25,.25),7).split(list(map(str,range(20))))
assert sum(map(len,a.values()))==20 and not(set(a['train'])&set(a['test']))
assert a==DatasetSplitter((.5,.25,.25),7).split(list(map(str,range(20))))
try: DatasetSplitter((.8,.3,-.1))
except ValueError: pass
else: raise AssertionError('Ratios invalides')
print('Tests corpus: OK')


Tests corpus: OK


## 5. Experiments

Comparer compte des documents avant/après et tailles des partitions.


In [3]:
for name,rows in [('brut',docs),('nettoyé',clean)]: print(name,len(rows),'documents')
print('Tailles:',{k:len(v) for k,v in splits.items()})


brut 5 documents
nettoyé 3 documents
Tailles: {'train': 2, 'validation': 0, 'test': 1}


## 6. Results

Bilan du corpus et tailles des ensembles.


In [4]:
{"bilan": report, "tailles": {k: len(v) for k, v in splits.items()}}


{'bilan': Report(total=5, kept=3, duplicates=1, short=1, mean_chars=26.666666666666668),
 'tailles': {'train': 2, 'validation': 0, 'test': 1}}

## 7. Discussion

La déduplication exacte ne détecte pas les paraphrases. Corpus réel : ajouter vérification langue/qualité et déduplication approchée.
